## Preprocessing -
includes, encoding, feature engineering, train-test split

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [4]:
df= pd.read_csv('../data/Telco-Customer-Churn.csv')
df.shape

(7043, 21)

1. Handling missing values

In [5]:
df['TotalCharges']= df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges']= df['TotalCharges'].astype(float)
df['TotalCharges']= df['TotalCharges'].fillna(0)

print(f"Missing val after cleaning: {df.isnull().sum().sum()}")

Missing val after cleaning: 0


2. Encoding Churn(Target Variable)

In [7]:
# text to numeric
df['Churn']= LabelEncoder().fit_transform(df['Churn'])

In [ ]:
df.info()

3. Encoding binary categorical var


In [8]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col]= LabelEncoder().fit_transform(df[col])

4. Encoding Nominal Categorical var

In [9]:
categorical_cols = ['Contract', 'PaymentMethod', 'InternetService', 'MultipleLines', 
                    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                    'TechSupport', 'StreamingTV', 'StreamingMovies']

df=pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding:{df.shape}")

Shape after encoding:(7043, 32)


In [10]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes
0,7590-VHVEG,0,0,1,0,1,0,1,29.85,29.85,...,False,True,False,False,False,False,False,False,False,False
1,5575-GNVDE,1,0,0,0,34,1,0,56.95,1889.50,...,False,False,False,True,False,False,False,False,False,False
2,3668-QPYBK,1,0,0,0,2,1,1,53.85,108.15,...,False,True,False,False,False,False,False,False,False,False
3,7795-CFOCW,1,0,0,0,45,0,0,42.30,1840.75,...,False,False,False,True,False,True,False,False,False,False
4,9237-HQITU,0,0,0,0,2,1,1,70.70,151.65,...,False,False,False,False,False,False,False,False,False,False


In [14]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges',
       'TotalCharges', 'Churn', 'Contract_One year', 'Contract_Two year',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'InternetService_Fiber optic', 'InternetService_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'AvgMonthlyCharges'],
      dtype='str')

5. Feature Engineering

In [11]:
## Avg monthly charges per tenure

df['AvgMonthlyCharges']= df['TotalCharges']/(df['tenure'] + 1)  # if tenure=0

In [15]:
# Total services count
service_cols = ['PhoneService', 'MultipleLines_Yes', 'InternetService_Fiber optic', 
                'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 
                'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes']

existing_services= [col for col in service_cols if col in df.columns]

df['TotalServices']= df[existing_services].sum(axis=1)

6. Split Features & Target(Churn)

In [17]:
X= df.drop('Churn', axis=1)
y= df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (7043, 33)
Target shape: (7043,)


7. Train-Test Split

In [18]:
X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}")
print(f"Test churn rate: {y_test.mean():.2%}")


Training set: (5634, 33)
Test set: (1409, 33)
Train churn rate: 26.54%
Test churn rate: 26.54%


8. Saving the Processed Data

In [24]:
# saved for modeling

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

pd.DataFrame(y_train).to_csv('../data/processed/y_train.csv', index= False)
pd.DataFrame(y_test).to_csv('../data/processed/y_test.csv', index= False)

In [25]:
# Save feature names
feature_names = X.columns.tolist()
pd.DataFrame(feature_names, columns=['feature']).to_csv('../data/processed/feature_names.csv', index=False)

print("Preprocessing completed. Processed data saved in '../data/processed/'.")

Preprocessing completed. Processed data saved in '../data/processed/'.
